In [ ]:
import kagglehub
import os
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q3-stage3-2026")

print("Path to dataset files:", path)

In [ ]:
def remap_mask(mask):
    # Remaps a mask's pixel values to a consecutive range starting at 0
    mask = mask.long()
    unique_values = torch.unique(mask)
    remapped_mask = torch.zeros_like(mask)

    for new_val, old_val in enumerate(sorted(unique_values.tolist())):
        remapped_mask[mask == old_val] = new_val

    return remapped_mask



In [ ]:
# TO DO
from torch.utils.data import Dataset, DataLoader, random_split

data_root   = "/kaggle/input/q3-stage3-2026/dataset"
images_dir  = os.path.join(data_root, "images")
masks_dir   = os.path.join(data_root, "masks")

print("Images dir:", images_dir)
print("Masks  dir:", masks_dir)
print("Sample images:", os.listdir(images_dir)[:5])
print("Sample masks :", os.listdir(masks_dir)[:5])

img_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.ToTensor(),
])

mask_transform = transforms.Compose([
    transforms.Resize((256, 256), interpolation=Image.NEAREST),
    transforms.PILToTensor(),
])

num_classes = 8

class SUIMDataset(Dataset):
    def __init__(self, images_dir, masks_dir, img_transform=None, mask_transform=None):
        self.images_dir = images_dir
        self.masks_dir  = masks_dir
        self.img_transform  = img_transform
        self.mask_transform = mask_transform

        self.image_files = sorted(os.listdir(images_dir))

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_name = self.image_files[idx]
        img_path = os.path.join(self.images_dir, img_name)

        mask_name = os.path.splitext(img_name)[0] + ".png"
        mask_path = os.path.join(self.masks_dir, mask_name)

        image = Image.open(img_path).convert("RGB")
        mask  = Image.open(mask_path)

        if self.img_transform:
            image = self.img_transform(image)
        if self.mask_transform:
            mask = self.mask_transform(mask)

        mask = mask.squeeze(0)
        mask = remap_mask(mask)
        return image, mask

full_dataset = SUIMDataset(images_dir, masks_dir, img_transform, mask_transform)
print("Full dataset size:", len(full_dataset))

train_size = int(0.8 * len(full_dataset))
val_size   = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=4, shuffle=False, num_workers=2, pin_memory=True)

print("Train size:", len(train_dataset))
print("Val size  :", len(val_dataset))


images, masks = next(iter(train_loader))

plt.figure(figsize=(8,8))
for i in range(4):
    img = images[i].permute(1,2,0).cpu().numpy()
    msk = masks[i].cpu().numpy()

    plt.subplot(4,2,2*i+1)
    plt.imshow(img)
    plt.title("Image")
    plt.axis("off")

    plt.subplot(4,2,2*i+2)
    plt.imshow(msk, cmap="tab20")
    plt.title("Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()



In [ ]:

# TO DO
!pip install -q segmentation-models-pytorch

import segmentation_models_pytorch as smp

model = smp.Unet(
    encoder_name="efficientnet-b1",
    encoder_weights="imagenet",
    in_channels=3,
    classes=num_classes,
)


In [ ]:
# TO DO
import torch.nn.functional as F

def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0

    for images, masks in dataloader:
        images = images.to(device)
        masks  = masks.to(device).long()

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, masks)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

    return running_loss / len(dataloader.dataset)


def validate_one_epoch(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0

    with torch.no_grad():
        for images, masks in dataloader:
            images = images.to(device)
            masks  = masks.to(device).long()

            outputs = model(images)
            loss = criterion(outputs, masks)
            running_loss += loss.item() * images.size(0)

    return running_loss / len(dataloader.dataset)


In [ ]:
# TO DO
import torch
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model  = model.to(device)

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

num_epochs = 10
train_losses, val_losses = [], []

for epoch in range(num_epochs):
    train_loss = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss   = validate_one_epoch(model, val_loader, criterion, device)

    train_losses.append(train_loss)
    val_losses.append(val_loss)

    print(f"Epoch {epoch+1}/{num_epochs} - "
          f"Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

plt.figure(figsize=(6,4))
plt.plot(train_losses, label="Train")
plt.plot(val_losses, label="Val")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Training & Validation Loss")
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
# TO DO
import numpy as np

model.eval()
images, masks = next(iter(val_loader))
images = images.to(device)
masks  = masks.to(device)

with torch.no_grad():
    outputs = model(images)
    preds   = torch.argmax(outputs, dim=1)

images  = images.cpu()
masks   = masks.cpu()
preds   = preds.cpu()

plt.figure(figsize=(9,9))
for i in range(3):
    img  = images[i].permute(1,2,0).numpy()
    gt   = masks[i].numpy()
    pr   = preds[i].numpy()

    plt.subplot(3,3,3*i+1)
    plt.imshow(img)
    plt.title("Image")
    plt.axis("off")

    plt.subplot(3,3,3*i+2)
    plt.imshow(gt, cmap="tab20")
    plt.title("GT Mask")
    plt.axis("off")

    plt.subplot(3,3,3*i+3)
    plt.imshow(pr, cmap="tab20")
    plt.title("Pred Mask")
    plt.axis("off")

plt.tight_layout()
plt.show()
